# 面试问题：Agent 的 Human-in-the-Loop 怎样设计，哪些动作必须审批，如何中断与恢复？

**一句话回答**：HITL 是确定性风险控制，不是让模型随意问“确认吗”。策略层根据动作影响、可逆性、金额、数据敏感和模型不确定性决定 allow/approve/deny；审批请求展示规范化参数与来源，token 绑定用户、动作摘要、过期和单次使用；Agent 持久化到 waiting，收到 approve/edit/reject 信号后按预期版本恢复。

本 Notebook 实现风险分类、参数绑定审批、优先队列、编辑语义、双人批准、急停、补偿和 HITL 指标。

In [ ]:
from dataclasses import dataclass
import hashlib, heapq, html, json, math

SEED120=12001
assert SEED120==12001
assert html.escape("<x>")=="&lt;x&gt;"
assert hashlib.sha256(b"approve").hexdigest()!=hashlib.sha256(b"reject").hexdigest()

## 1. 风险策略输入来自宿主事实

关键维度是 read/write/destructive、可逆性、金额、外发、敏感数据、目标身份和不确定性。模型可提供解释，但工具风险、账户 scope 与金额由确定性代码解析。只读低风险可自动，删除/支付/发布通常审批或禁止。

In [ ]:
@dataclass(frozen=True)
class Action120: action_id:str; kind:str; amount:float=0.; external:bool=False; sensitive:bool=False; reversible:bool=True; confidence:float=1.
def risk_decision120(a):
    if a.kind in {"delete_account","export_secret"}: return "deny"
    if a.amount>=100 or a.external or a.sensitive or not a.reversible or a.confidence<.7: return "approve"
    return "allow"
assert risk_decision120(Action120("a1","read"))=="allow"
assert risk_decision120(Action120("a2","transfer",200))=="approve"
assert risk_decision120(Action120("a3","delete_account",reversible=False))=="deny"

## 2. 审批页展示规范化动作，不展示模型自述

请求包含谁、对什么资源、做什么、具体参数、数据来源、风险、可逆性和过期时间。所有外部文本按 UI 上下文编码，并标记 taint；用户看到的 digest 与执行器校验的是同一规范化 payload，防止审批后参数偷换。

In [ ]:
def canonical_action120(action,args): return json.dumps({"action_id":action.action_id,"kind":action.kind,"args":args},sort_keys=True,separators=(",",":"),ensure_ascii=False)
def approval_view120(action,args,tainted=()):
    canonical=canonical_action120(action,args); return {"title":f"批准 {html.escape(action.kind)}","rows":[{"key":k,"value":html.escape(str(v)),"untrusted":k in tainted} for k,v in sorted(args.items())],"digest":hashlib.sha256(canonical.encode()).hexdigest(),"decision":risk_decision120(action)}
pay120=Action120("pay-1","transfer",200,True,False,False,.9); view120=approval_view120(pay120,{"to":"u2","note":"<urgent>"},{"note"})
assert view120["decision"]=="approve"
assert next(r for r in view120["rows"] if r["key"]=="note")["value"]=="&lt;urgent&gt;"
assert len(view120["digest"])==64

## 3. Approval token 绑定身份、digest、版本、TTL 和一次性消费

token 不能只绑定 run ID；同一 run 后续重规划可能产生不同金额。执行时重新鉴权并比较 digest、expected state version、审批者与发起者规则。过期、重复、参数变化或状态已前进全部拒绝。

In [ ]:
approvals120={}
def issue120(approver,user,digest,expires,state_version):
    token=hashlib.sha256(f"{approver}:{user}:{digest}:{expires}:{state_version}".encode()).hexdigest(); approvals120[token]={"approver":approver,"user":user,"digest":digest,"expires":expires,"state_version":state_version,"used":False}; return token
def consume120(token,user,digest,now,state_version):
    a=approvals120.get(token)
    if not a or a["used"] or a["user"]!=user or a["digest"]!=digest or now>a["expires"] or a["state_version"]!=state_version: return False
    a["used"]=True; return True
token120=issue120("reviewer","u1",view120["digest"],100,4)
assert consume120(token120,"u1",view120["digest"],99,4)
assert not consume120(token120,"u1",view120["digest"],99,4)
assert not consume120(issue120("r","u1",view120["digest"],100,4),"u1","different",99,4)

## 4. 审批队列要有优先级、SLA 与租户公平

高影响/临近 deadline 优先，但不能让单一租户淹没 reviewer。队列项带 created、expires 和 escalation target；超时默认策略按风险 fail-closed 或安全降级，不能默认批准。下面使用确定性优先键。

In [ ]:
queue120=[]
def enqueue120(item): heapq.heappush(queue120,(-item["risk"],item["expires"],item["tenant"],item["id"]))
enqueue120({"id":"q1","risk":5,"expires":50,"tenant":"T1"}); enqueue120({"id":"q2","risk":3,"expires":20,"tenant":"T2"}); enqueue120({"id":"q3","risk":5,"expires":40,"tenant":"T2"})
first_q120=heapq.heappop(queue120); second_q120=heapq.heappop(queue120)
assert first_q120[-1]=="q3"
assert second_q120[-1]=="q1"
assert len(queue120)==1

## 5. Approve、Edit、Reject 是不同信号

approve 执行完全相同动作；reject 终止或要求新计划；edit 不是对旧动作的批准，而是创建新 proposal、新 digest 并重新经过策略。自由文本反馈作为数据进入 planner，不能被解释为隐藏权限授予。

In [ ]:
def apply_signal120(state,signal):
    if state["status"]!="waiting" or signal["expected_version"]!=state["version"]: return state,"stale_signal"
    out=dict(state); out["version"]+=1
    if signal["kind"]=="approve": out["status"]="ready"; return out,"approved"
    if signal["kind"]=="reject": out["status"]="rejected"; return out,"rejected"
    if signal["kind"]=="edit": out["status"]="replan"; out["edited_args"]=signal["args"]; return out,"new_proposal_required"
    return state,"unknown_signal"
waiting120={"status":"waiting","version":7,"action":"pay-1"}; approved_state120,msg120=apply_signal120(waiting120,{"kind":"approve","expected_version":7})
assert approved_state120["status"]=="ready" and msg120=="approved"
assert apply_signal120(waiting120,{"kind":"approve","expected_version":6})[1]=="stale_signal"
assert apply_signal120(waiting120,{"kind":"edit","expected_version":7,"args":{"amount":50}})[1]=="new_proposal_required"

## 6. 高风险动作采用职责分离与 quorum

大额支付、生产发布、密钥导出可要求两名不同 reviewer，且发起者不能自批。聚合器按 action digest 收集签名；任一拒绝是否立即否决由政策定义。多个“同一模型代理人”不构成人类独立审批。

In [ ]:
votes120={}
def vote120(digest,reviewer,decision,initiator,required=2):
    if reviewer==initiator: return "self_approval_denied"
    votes120.setdefault(digest,{})[reviewer]=decision; v=votes120[digest]
    if "reject" in v.values(): return "rejected"
    return "approved" if sum(x=="approve" for x in v.values())>=required else "pending"
d120=view120["digest"]
assert vote120(d120,"u1","approve","u1")=="self_approval_denied"
assert vote120(d120,"r1","approve","u1")=="pending"
assert vote120(d120,"r2","approve","u1")=="approved"

## 7. 急停、撤销和补偿不能依赖模型响应

用户 cancel 直接到宿主状态机，停止派发新动作并传播取消。可逆操作优先 staged/soft delete，并保留 undo TTL；不可逆动作执行前提高审批级别。已经提交的跨系统动作通过 saga 补偿，急停不是时间倒流。

In [ ]:
def emergency_stop120(run):
    out=dict(run); out["cancelled"]=True; out["status"]="cancelling"; out["pending"]=[]; return out
run120={"status":"running","cancelled":False,"pending":["a","b"],"completed":["reserve","draft"]}; stopped120=emergency_stop120(run120)
compensation120={"reserve":"release","draft":"discard"}; undo120=[compensation120[x] for x in reversed(stopped120["completed"]) if x in compensation120]
assert stopped120["cancelled"] and stopped120["pending"]==[]
assert undo120==["discard","release"]
assert run120["pending"]==["a","b"]

## 8. HITL 评测不只看“拦了多少”

指标包括危险动作自动放行率、无谓审批率、审批等待 p95、参数理解错误、edit/reject 比例、过期与重复 token、取消生效时间。用受控攻击与正常流量共同评估；过多审批会造成 fatigue，最终让用户机械点击通过。

In [ ]:
cases120=[("danger","approve"),("safe","allow"),("safe","approve"),("danger","deny"),("safe","allow")]
dangerous_auto120=sum(kind=="danger" and decision=="allow" for kind,decision in cases120); needless120=sum(kind=="safe" and decision=="approve" for kind,decision in cases120)/sum(kind=="safe" for kind,_ in cases120)
manifest120={"schema":1,"risk_policy":"v4","approval":"identity+digest+ttl+state","high_risk_quorum":2,"edit":"new_proposal","timeout":"fail_closed","cancel":"host_signal+saga"}; digest_manifest120=hashlib.sha256(json.dumps(manifest120,sort_keys=True).encode()).hexdigest()
assert dangerous_auto120==0
assert math.isclose(needless120,1/3)
assert len(digest_manifest120)==64 and manifest120["edit"]=="new_proposal"

## 面试总结

强回答是：**确定性风险分级 → 规范化审批视图 → identity/digest/TTL/version 绑定 token → SLA/公平队列 → approve/edit/reject 明确语义 → 职责分离 quorum → 宿主急停与补偿 → 安全和 fatigue 联合评测**。HITL 的关键是人真正控制具体动作，而不是增加一个装饰性弹窗。

延伸阅读：[OWASP Excessive Agency](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/)、[MCP Security Principles](https://modelcontextprotocol.io/specification/2025-03-26/index#security-and-trust-safety)、[Trustworthy Agents](https://www.anthropic.com/research/trustworthy-agents)。